In [ ]:
!pip install -q gradio torchio SimpleITK


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.6/203.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.4 MB/s eta 0:00:00


# 1. Mount Google Drive if not already mounted

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.ndimage as ndimage
import cv2
from google.colab import drive


try:
    drive.mount('/content/drive')
except:
    pass

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Mounted at /content/drive


# ----------------------------------------------------
# 2. Re-Define Model Architecture
# ----------------------------------------------------

In [ ]:
class CNN3DBranch(nn.Module):
    def __init__(self, output_dim=128):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            nn.Conv3d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool3d(2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )
    def forward(self, x):
        return self.fc(self.conv_blocks(x))

class TabularMLPBranch(nn.Module):
    def __init__(self, input_dim=5, output_dim=32):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 16), nn.ReLU(),
            nn.Linear(16, output_dim), nn.ReLU()
        )
    def forward(self, x):
        return self.mlp(x)

class HybridGatedFusionModel(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.cnn = CNN3DBranch(output_dim=128)
        self.mlp = TabularMLPBranch(input_dim=5, output_dim=32)
        self.gate = nn.Sequential(nn.Linear(128 + 32, 128 + 32), nn.Sigmoid())
        self.classifier = nn.Sequential(
            nn.Linear(128 + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    def forward(self, image, tabular):
        img_features = self.cnn(image)
        tab_features = self.mlp(tabular)
        combined = torch.cat((img_features, tab_features), dim=1)
        gated_features = combined * self.gate(combined)
        return self.classifier(gated_features)

# ----------------------------------------------------
# 3. Load Trained Ensemble Weights
# ----------------------------------------------------

In [ ]:
ensemble_models = []
for fold in range(1, 6):
    model = HybridGatedFusionModel(num_classes=2).to(device)
    # Check for full-cohort weights first, fallback to hybrid weights
    path_fc = f"/content/drive/MyDrive/Lung_Nodule_Project/processed_patches/full_cohort_model_fold{fold}.pt"
    path_hybrid = f"/content/drive/MyDrive/Lung_Nodule_Project/models/hybrid_model_fold{fold}.pt"
    try:
        model.load_state_dict(torch.load(path_fc, map_location=device))
    except:
        model.load_state_dict(torch.load(path_hybrid, map_location=device))
    model.eval()
    ensemble_models.append(model)

# ----------------------------------------------------
# 4. 3D Grad-CAM Generator
# ----------------------------------------------------

In [ ]:
class GradCAM3D:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate_cam(self, image_tensor, tabular_tensor, target_class=1):
        self.model.eval()
        self.model.zero_grad()
        output = self.model(image_tensor, tabular_tensor)
        score = output[0, target_class]
        score.backward()

        gradients = self.gradients[0].cpu().data.numpy()
        activations = self.activations[0].cpu().data.numpy()
        weights = np.mean(gradients, axis=(1, 2, 3))

        cam = np.zeros(activations.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * activations[i]

        cam = np.maximum(cam, 0)
        zoom_factors = (
            image_tensor.shape[2] / cam.shape[0],
            image_tensor.shape[3] / cam.shape[1],
            image_tensor.shape[4] / cam.shape[2]
        )
        cam_upsampled = ndimage.zoom(cam, zoom_factors, order=1)
        cam_upsampled = cam_upsampled - np.min(cam_upsampled)
        cam_upsampled = cam_upsampled / (np.max(cam_upsampled) + 1e-8)
        return cam_upsampled

# ----------------------------------------------------
# 5. Load Manifest & Setup Inference Engine
# ----------------------------------------------------

In [ ]:
manifest_df = pd.read_csv("/content/drive/MyDrive/Lung_Nodule_Project/processed_patches/manifest.csv")
nodule_ids = manifest_df['nodule_id'].tolist()

OPTIMAL_THRESHOLD = 0.5783  # Derived via Youden's J in Phase 3 evaluation

def run_ai_prediction(selected_nodule_id, subtlety, sphericity, margin, spiculation, texture):
    row = manifest_df[manifest_df['nodule_id'] == selected_nodule_id].iloc[0]
    patch_data = torch.load(row['patch_path'], weights_only=False)

    # 3D Tensor
    image_tensor = patch_data['tensor'].unsqueeze(0).to(device)
    image_tensor.requires_grad = True

    # Clinical tabular features
    tabs = np.array([subtlety, sphericity, margin, spiculation, texture], dtype=np.float32)
    tab_tensor = torch.tensor(tabs).unsqueeze(0).to(device)

    # Ensemble Inference
    batch_probs = torch.zeros(1, 2).to(device)
    with torch.no_grad():
        for m in ensemble_models:
            logits = m(image_tensor, tab_tensor)
            batch_probs += F.softmax(logits, dim=1)

    avg_probs = (batch_probs / 5.0).cpu().numpy()[0]
    malignant_prob = float(avg_probs[1])
    is_malignant = malignant_prob >= OPTIMAL_THRESHOLD

    prediction_text = f"**Status:** {'MALIGNANT' if is_malignant else 'BENIGN'}\n\n" \
                      f"**Consensus Malignancy Score:** {malignant_prob:.2%}\n" \
                      f"**Calibrated Decision Threshold:** {OPTIMAL_THRESHOLD:.2%}"

    # Generate 3D Grad-CAM
    cam_tool = GradCAM3D(ensemble_models[0], ensemble_models[0].cnn.conv_blocks[9])
    cam3d = cam_tool.generate_cam(image_tensor, tab_tensor, target_class=1 if is_malignant else 0)

    vol = image_tensor.squeeze().cpu().detach().numpy()
    mid_z, mid_y, mid_x = vol.shape[0]//2, vol.shape[1]//2, vol.shape[2]//2

    # Render Visual Plot
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # Axial
    axes[0].imshow(vol[mid_z, :, :], cmap='gray')
    axes[0].imshow(cam3d[mid_z, :, :], cmap='jet', alpha=0.45)
    axes[0].set_title("Axial Slice")
    axes[0].axis('off')

    # Coronal
    axes[1].imshow(vol[:, mid_y, :], cmap='gray')
    axes[1].imshow(cam3d[:, mid_y, :], cmap='jet', alpha=0.45)
    axes[1].set_title("Coronal Slice")
    axes[1].axis('off')

    # Sagittal
    axes[2].imshow(vol[:, :, mid_x], cmap='gray')
    axes[2].imshow(cam3d[:, :, mid_x], cmap='jet', alpha=0.45)
    axes[2].set_title("Sagittal Slice")
    axes[2].axis('off')

    plt.tight_layout()
    return prediction_text, fig

# ----------------------------------------------------
# 6. Launch Gradio Dashboard
# ----------------------------------------------------

In [ ]:
demo = gr.Interface(
    fn=run_ai_prediction,
    inputs=[
        gr.Dropdown(choices=nodule_ids[:50], value=nodule_ids[0], label="Select Nodule ID"),
        gr.Slider(1.0, 5.0, value=3.0, step=0.1, label="Subtlety"),
        gr.Slider(1.0, 5.0, value=3.0, step=0.1, label="Sphericity"),
        gr.Slider(1.0, 5.0, value=3.0, step=0.1, label="Margin"),
        gr.Slider(1.0, 5.0, value=3.0, step=0.1, label="Spiculation"),
        gr.Slider(1.0, 5.0, value=3.0, step=0.1, label="Texture"),
    ],
    outputs=[
        gr.Markdown(label="Diagnostic Assessment"),
        gr.Plot(label="3D Grad-CAM Explainable AI Heatmaps")
    ],
    title="Hybrid Explainable AI: Lung Nodule Malignancy Detection",
    description="Multimodal lung nodule evaluation combining 3D CT feature maps with calibrated radiomic gating."
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ee1ee6c45feabc7ef4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Model Performance Summary

Metric	Result	Description
Ensemble ROC-AUC	0.9042	High diagnostic discrimination across 5 cross-validation folds.
Youden’s J Optimal Threshold	0.5783	Balances clinical sensitivity with false-positive minimization.
Overall Accuracy	86.00%	Evaluated across 1,242 consensus-labeled nodules.
Benign Precision / Recall / F1	0.89 / 0.90 / 0.90	High specificity for non-malignant scans.
Malignant Precision / Recall / F1	0.77 / 0.76 / 0.77	Solid detection of malignant nodules despite class imbalance.

Ablation Study

Model Configuration	ROC-AUC	Interpretation
3D CNN Only	0.6901	Demonstrates image spatial features alone are insufficient.
Tabular MLP Only	0.8985	Strong baseline using handcrafted radiomic parameters.
Hybrid Gated Fusion	0.9042	Proves multimodal synergy yields superior classification.